# Quick start: SPY max pain, gamma exposure and walls

The daily option series of this dataset, loaded and put on one scale. More tickers (every optionable US stock and ETF we track): https://spyforecast.com/historical-options-data/

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

# the folder of this dataset: here, or under /kaggle/input on Kaggle
base = next((d for d in ["data", *glob.glob("/kaggle/input/*/data"), *glob.glob("/kaggle/input/*")] if os.path.exists(os.path.join(d, "SPY.csv"))), "data")
df = pd.read_csv(os.path.join(base, "SPY.csv"), parse_dates=["date"]).set_index("date")
LEVELS = ["max_pain_front", "max_pain_monthly", "call_wall_near", "put_wall_near", "gamma_flip"]
for c in LEVELS:
    df[c] = df[c] * df["split_factor"]   # strike levels are raw strikes; the close is split-adjusted
df[["close", *LEVELS, "gex", "pc_oi_ratio", "iv30"]].tail()

## The last year: the close with its walls, gamma flip and monthly max pain

In [ ]:
y = df[df.index >= df.index.max() - pd.Timedelta(days=365)]
ax = y["close"].plot(figsize=(12, 5), lw=1.6, label="close")
for c, st in [("call_wall_near", "--"), ("put_wall_near", "--"), ("gamma_flip", ":"), ("max_pain_monthly", "-.")]:
    y[c].plot(ax=ax, ls=st, lw=1, label=c)
ax.legend(); ax.set_title("SPY: close, walls, gamma flip, monthly max pain"); plt.show()

## A first question: is the next day calmer when dealers are long gamma?

`gamma_regime` is +1 when dealers are net long gamma (their hedging leans against moves) and −1 when short. Compare the size of the next session's move.

In [ ]:
d = df.copy()
d["next_abs_move_pct"] = d["close"].pct_change().shift(-1).abs() * 100
d = d[d["chain_flag"] == 0].dropna(subset=["gamma_regime", "next_abs_move_pct"])
d.groupby("gamma_regime")["next_abs_move_pct"].describe()[["count", "mean", "50%"]]

## Is that just high implied volatility?

Short gamma shows up more when IV is already high. Split the same days into five groups by 30-day ATM IV and compare inside each group (rows with no usable IV are left out).

In [ ]:
d2 = d[d["iv30"] > 0.03].copy()
d2["iv_group"] = pd.qcut(d2["iv30"], 5)
d2.pivot_table(index="iv_group", columns="gamma_regime", values="next_abs_move_pct", aggfunc="mean", observed=True).round(2)

## Against the move IV priced in

The average absolute daily move a 30-day IV implies is IV / √252 × √(2/π). Ratio of the actual next-day move to it, by gamma regime (1.0 = the move IV priced).

In [ ]:
d2["priced_move_pct"] = d2["iv30"] / 252 ** 0.5 * 100 * (2 / 3.141592653589793) ** 0.5
(d2["next_abs_move_pct"] / d2["priced_move_pct"]).groupby(d2["gamma_regime"]).mean().round(3)

## Put/call ratio and implied volatility over the whole history

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
df["pc_oi_ratio"].rolling(20).mean().plot(ax=ax[0], title="Put/call open interest ratio (20-session mean)")
(df["iv30"] * 100).plot(ax=ax[1], title="30-day implied volatility, %")
plt.tight_layout(); plt.show()

Every column: `columns.csv`. The same series for every ticker, and today's levels: https://spyforecast.com/historical-options-data/